# Datagen

Generate realistic Delta tables and a Direct Lake semantic model from a Power BI `.vpax` file.

## Setup

1. **Attach a Lakehouse** — click the Lakehouse icon in the left sidebar and select (or create) a Lakehouse
2. **Upload your `.vpax` file** — place it in `Files/datagen/` in the attached Lakehouse
   - Export a `.vpax` from [DAX Studio](https://daxstudio.org/) → Advanced → Export Metrics
3. **Run Cell 1** below to install datagen (auto-downloads from GitHub if needed)
4. **Edit Cell 2** — change the `.vpax` filename to match yours
5. **Run Cell 2** — generates Delta tables, deploys a semantic model, and prints a comparison report


In [ ]:
# Install datagen — prefers the newest local .whl in the notebook's builtin resources, otherwise downloads the latest GitHub release
import glob, re, subprocess, sys, os, tempfile, urllib.request, json

USE_LATEST = True  # Set False to skip the GitHub check entirely (local .whl only)

def _ver_tuple(s):
    return tuple(int(p) for p in re.findall(r'\d+', s))

def _local_whl():
    # Look in notebook builtin resources first, then fall back to the lakehouse Files/datagen folder
    candidates = []
    for d in ("builtin", "/lakehouse/default/Files/datagen"):
        if os.path.isdir(d):
            candidates.extend(glob.glob(f"{d}/datagen_fabric-*.whl"))
    if not candidates:
        return None, None
    def ver_of(p):
        m = re.search(r'datagen_fabric-([\d.]+)', os.path.basename(p))
        return _ver_tuple(m.group(1)) if m else (0,)
    best = max(candidates, key=ver_of)
    return best, ver_of(best)

def _release_whl():
    api_url = "https://api.github.com/repos/dbrownems/datagen/releases/latest"
    with urllib.request.urlopen(api_url) as resp:
        release = json.loads(resp.read())
    asset = next(a for a in release["assets"] if a["name"].endswith(".whl"))
    local = os.path.join(tempfile.gettempdir(), asset['name'])
    if not os.path.exists(local):
        print(f"Downloading {asset['name']} ...")
        urllib.request.urlretrieve(asset["browser_download_url"], local)
    return local, _ver_tuple(asset['name'])

local_whl, local_ver = _local_whl()
if USE_LATEST:
    rel_whl, rel_ver = _release_whl()
    if local_whl and local_ver >= rel_ver:
        whl = local_whl
        print(f"Using local v{'.'.join(map(str, local_ver))} (>= released v{'.'.join(map(str, rel_ver))})")
    else:
        whl = rel_whl
else:
    whl = local_whl or _release_whl()[0]

print(f"Installing {os.path.basename(whl)}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", whl, "semantic-link-labs",
    "-q", "--no-warn-conflicts", "--disable-pip-version-check", "--force-reinstall", "--no-deps",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "semantic-link-labs", "-q", "--no-warn-conflicts", "--disable-pip-version-check"])
import datagen; print(f"Ready — datagen v{datagen.__version__}")

# Verify the installed version matches the loaded version
import importlib.metadata
installed = importlib.metadata.version('datagen-fabric')
loaded = datagen.__version__
if installed != loaded:
    raise RuntimeError(
        f'Version mismatch: installed v{installed} but loaded v{loaded}. '
        f'Please restart the Spark session (Session → Stop session) and re-run.'
    )


## Generate

Edit the `.vpax` filename below, then run the cell. This will:

1. **Parse** the `.vpax` and infer data distributions from column statistics
2. **Generate** Delta tables matching the original row counts and cardinality
3. **Deploy** a Direct Lake semantic model with all measures, relationships, and column metadata
4. **Compare** the generated tables against the expected statistics and print a report

### Options for `generate(spark, vpax_path, ...)`

| Parameter | Default | Description |
|---|---|---|
| `vpax_path` | _required_ | Path to the `.vpax` file. |
| `output_path` | `"Tables/"` | Where Delta tables are written, relative to the attached lakehouse. |
| `output_format` | `"delta"` | `"delta"` or `"parquet"`. |
| `mode` | `"import"` | `"import"` (default, Power Query import mode) or `"direct_lake"`. |
| `seed` | `42` | Random seed for reproducible generation. |
| `deploy_model` | `True` | If `False`, skip semantic model deployment (tables only). |
| `compare` | `True` | If `True`, run a post-generation accuracy report comparing generated row counts and column cardinalities against the `.vpax` expectations. Returns it as a pandas DataFrame. |
| `overwrite_tables` | `True` | If `False`, skip tables that already exist and only generate missing ones. |
| `overwrite_model` | `True` | If `False`, fail when a semantic model with the target name already exists instead of replacing it. |
| `dataset` | _vpax model name_ | Override the deployed semantic model name. |
| `workspace` | _current_ | Target Fabric workspace for the semantic model. |
| `lakehouse` | _attached_ | Source lakehouse for Direct Lake. |
| `include_hidden` | `False` | Include hidden columns/tables from the VPAX. |
| `include_calculated` | `False` | Include calculated columns. |
| `queries_path` | `None` | Optional path to a `.jsonl` DAX query trace; literal predicate values are extracted and used to seed low-cardinality columns so generated data contains the values queries reference. |
| `skew_recent` | `False` | If `True`, skew date-distributed numeric/fact columns toward more recent dates rather than uniform across the full date range. |
| `replace_username_with_customdata` | `False` | If `True`, rewrite `USERNAME()` and `USERPRINCIPALNAME()` in measures and RLS row filters to `CUSTOMDATA()`. Useful for load testing where the tester impersonates many users via the CustomData connection property. |


In [ ]:
from datagen import generate

VPAX_FILE = ""  # ← set your .vpax filename, or leave blank to auto-detect

# Auto-detect: use the first .vpax file in the datagen folder
if not VPAX_FILE:
    vpax_files = sorted(glob.glob(f"{DATAGEN_DIR}/*.vpax"))
    if not vpax_files:
        raise FileNotFoundError(f"No .vpax files found in {DATAGEN_DIR}. Upload one and re-run.")
    vpax_path = vpax_files[0]
    print(f"Auto-detected: {os.path.basename(vpax_path)}")
else:
    vpax_path = f"{DATAGEN_DIR}/{VPAX_FILE}"

report = generate(
    spark,
    vpax_path,
    compare=False,
    overwrite_tables=False,
    overwrite_model=True,
)